# 03 - Treinamento do Modelo de Predição de Obesidade

Este notebook realiza o treinamento, comparação, otimização e validação de modelos de classificação multiclasse para predição do nível de obesidade. O objetivo mínimo é superar 75% de acurácia no conjunto de teste.

## 1. Importação das bibliotecas

São utilizadas bibliotecas de manipulação de dados, pré-processamento, modelagem, otimização de hiperparâmetros, avaliação e persistência do modelo.

In [1]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='Set2')

RANDOM_STATE = 42
DATA_PATH = Path('../data/obesity_feature_engineered.csv')
RAW_DATA_PATH = Path('../data/obesity.csv')
MODEL_PATH = Path('../models/melhor_modelo_obesidade.pkl')

NOISY_COLUMNS = ['FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
TARGET_CANDIDATES = ['Obesity_level', 'Obesity']

## 2. Carga dos dados

O notebook tenta carregar primeiro o arquivo tratado no passo de Feature Engineering (`obesity_feature_engineered.csv`). Caso ele ainda não exista, carrega `obesity.csv` e aplica o arredondamento das colunas com ruído decimal.

In [ ]:
def round_noisy_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Arredonda colunas com ruídos decimais para o inteiro mais próximo."""
    df = df.copy()

    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').round().astype('Int64')

    return df


def load_modeling_dataset(processed_path: Path, raw_path: Path) -> pd.DataFrame:
    """Carrega a base tratada ou aplica o tratamento mínimo na base bruta."""
    if processed_path.exists():
        print(f'Carregando base tratada: {processed_path}')
        return pd.read_csv(processed_path)

    if raw_path.exists():
        print(f'Base tratada não encontrada. Carregando base bruta: {raw_path}')
        df = pd.read_csv(raw_path)
        return round_noisy_columns(df, NOISY_COLUMNS)

    raise FileNotFoundError('Nenhum arquivo de dados encontrado em ../data/.')


def identify_target_column(df: pd.DataFrame, candidates: list[str]) -> str:
    """Identifica a coluna alvo entre os nomes possíveis."""
    for col in candidates:
        if col in df.columns:
            return col

    raise ValueError(f'Nenhuma coluna alvo encontrada. Esperado: {candidates}')


df = load_modeling_dataset(DATA_PATH, RAW_DATA_PATH)
target_col = identify_target_column(df, TARGET_CANDIDATES)

print(f'Dimensões da base: {df.shape}')
print(f'Coluna alvo: {target_col}')
display(df.head())

## 3. Separação entre features e alvo

A variável alvo é codificada com `LabelEncoder`, enquanto as variáveis explicativas serão tratadas dentro de um `Pipeline` do scikit-learn para evitar vazamento de dados.

In [ ]:
df_model = df.drop_duplicates().copy()

X = df_model.drop(columns=[target_col])
y = df_model[target_col]

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

target_mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))
print('Mapeamento da variável alvo:')
display(target_mapping)

numeric_features = X.select_dtypes(include=['int64', 'float64', 'Int64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print('Features numéricas:', numeric_features)
print('Features categóricas:', categorical_features)

## 4. Divisão em treino e teste

A divisão holdout é estratificada para preservar a proporção das classes de obesidade nos conjuntos de treino e teste.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

stratification_check = pd.DataFrame({
    'train_distribution': pd.Series(y_train).value_counts(normalize=True).sort_index(),
    'test_distribution': pd.Series(y_test).value_counts(normalize=True).sort_index()
})

display(stratification_check)

## 5. Pipeline de pré-processamento

O pré-processamento inclui `StandardScaler` para variáveis numéricas e `OneHotEncoder` para variáveis categóricas. Esse bloco será reutilizado nos pipelines dos modelos.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ],
    remainder='drop'
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 6. Comparação inicial de modelos

Serão comparados dois modelos robustos para classificação multiclasse: `RandomForestClassifier` e `XGBClassifier`. A comparação usa Macro F1-Score em validação cruzada por ser uma métrica adequada quando há múltiplas classes.

In [ ]:
models = {
    'RandomForest': Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', RandomForestClassifier(
                random_state=RANDOM_STATE,
                class_weight='balanced',
                n_jobs=-1
            ))
        ]
    ),
    'XGBoost': Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', XGBClassifier(
                objective='multi:softprob',
                eval_metric='mlogloss',
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method='hist'
            ))
        ]
    )
}

baseline_results = []

for model_name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')

    baseline_results.append({
        'model': model_name,
        'accuracy': accuracy,
        'macro_f1': macro_f1
    })

baseline_results_df = pd.DataFrame(baseline_results).sort_values('macro_f1', ascending=False)
display(baseline_results_df)

## 7. Otimização leve de hiperparâmetros

A otimização abaixo utiliza `GridSearchCV` com grades leves para manter o treinamento viável, buscando o melhor modelo pela métrica `f1_macro`.

In [ ]:
param_grids = {
    'RandomForest': {
        'model__n_estimators': [200, 400],
        'model__max_depth': [None, 12, 20],
        'model__min_samples_split': [2, 5],
        'model__min_samples_leaf': [1, 2],
    },
    'XGBoost': {
        'model__n_estimators': [200, 400],
        'model__max_depth': [3, 5],
        'model__learning_rate': [0.05, 0.10],
        'model__subsample': [0.8, 1.0],
        'model__colsample_bytree': [0.8, 1.0],
    }
}

search_results = []
best_searches = {}

for model_name, pipeline in models.items():
    print(f'Executando GridSearchCV para {model_name}...')

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)
    best_searches[model_name] = search

    y_pred = search.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')

    search_results.append({
        'model': model_name,
        'best_cv_macro_f1': search.best_score_,
        'test_accuracy': accuracy,
        'test_macro_f1': macro_f1,
        'best_params': search.best_params_
    })

search_results_df = pd.DataFrame(search_results).sort_values('test_macro_f1', ascending=False)
display(search_results_df)

## 8. Seleção do modelo vencedor

O modelo vencedor é escolhido pelo maior Macro F1-Score no conjunto de teste. Em seguida, validamos se a acurácia ou o Macro F1-Score superou o objetivo mínimo de 75%.

In [ ]:
winner_name = search_results_df.iloc[0]['model']
best_model = best_searches[winner_name].best_estimator_

print(f'Modelo vencedor: {winner_name}')
print('Melhores hiperparâmetros:')
display(best_searches[winner_name].best_params_)

y_pred = best_model.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred)
final_macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f'Accuracy no teste: {final_accuracy:.4f}')
print(f'Macro F1-Score no teste: {final_macro_f1:.4f}')

if final_accuracy >= 0.75 or final_macro_f1 >= 0.75:
    print('Meta atingida: Accuracy ou Macro F1-Score superior a 75%.')
else:
    print('Atenção: meta de 75% não atingida. Avalie ampliar a busca de hiperparâmetros ou revisar as features.')

## 9. Relatório de classificação

O relatório abaixo detalha precisão, recall e F1-Score por classe, permitindo avaliar o desempenho do modelo em cada nível de obesidade.

In [ ]:
report = classification_report(
    y_test,
    y_pred,
    target_names=target_encoder.classes_,
    digits=4
)

print(report)

report_df = pd.DataFrame(
    classification_report(
        y_test,
        y_pred,
        target_names=target_encoder.classes_,
        output_dict=True
    )
).T

display(report_df)

## 10. Matriz de confusão

A matriz de confusão mostra os acertos e erros do modelo por classe real e classe predita.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=target_encoder.classes_,
    yticklabels=target_encoder.classes_
)
plt.title(f'Matriz de Confusão - {winner_name}')
plt.xlabel('Classe predita')
plt.ylabel('Classe real')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Exportação do modelo final

O pipeline final contém o pré-processamento completo, incluindo scaler e encoder de features. Também salvamos o `LabelEncoder` da variável alvo e metadados de avaliação no mesmo arquivo `.pkl` usando `joblib`.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

model_artifact = {
    'model_name': winner_name,
    'pipeline': best_model,
    'target_encoder': target_encoder,
    'target_mapping': target_mapping,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'target_column': target_col,
    'metrics': {
        'test_accuracy': final_accuracy,
        'test_macro_f1': final_macro_f1,
    },
    'best_params': best_searches[winner_name].best_params_,
}

joblib.dump(model_artifact, MODEL_PATH)

print(f'Modelo final salvo em: {MODEL_PATH}')

## 12. Como carregar o modelo salvo

A célula abaixo demonstra como recarregar o artefato treinado para uso posterior na aplicação Streamlit.

In [ ]:
loaded_artifact = joblib.load(MODEL_PATH)

print('Artefato carregado com sucesso.')
print('Modelo:', loaded_artifact['model_name'])
print('Métricas:', loaded_artifact['metrics'])